In [ ]:
import os, sys, subprocess, zipfile, shutil
from pathlib import Path

os.chdir("/kaggle/working")
TASK = "severity"  # change to "poor_dilation" for the other task
MODELS = ["resnet18"]  # first run; use ["all"] for all 10 models
RUN_NAME = "original_ids_v1"
assert TASK in {"severity", "poor_dilation"}

# Upload the supplied ZIP as a Kaggle input first.
archives = list(Path("/kaggle/input").rglob("optuna_complete.zip"))
PROJECT_DIR = Path("/kaggle/working/optuna_complete")
if archives:
    assert len(archives) == 1, f"Multiple project ZIPs found: {archives}"
    with zipfile.ZipFile(archives[0]) as z:
        for name in z.namelist():
            target = (Path("/kaggle/working") / name).resolve()
            assert target.is_relative_to(Path("/kaggle/working")), name
        z.extractall("/kaggle/working")
else:
    # Kaggle may extract the ZIP when creating the input dataset.
    candidates = [p.parent for p in Path("/kaggle/input").rglob("main.py")
                  if (p.parent / "metadata_utils.py").is_file()
                  and (p.parent / "Kaggle_Optuna.ipynb").is_file()]
    assert len(candidates) == 1, f"Attach the supplied project input; candidates: {candidates}"
    shutil.copytree(candidates[0], PROJECT_DIR, dirs_exist_ok=True)
assert (PROJECT_DIR / "main.py").is_file()
print("Project:", PROJECT_DIR)


In [ ]:
subprocess.run([
    sys.executable, "-m", "pip", "install", "-r",
    str(PROJECT_DIR / "requirements-kaggle.txt")
], check=True)

# Keep Kaggle's existing torch/torchvision installation.
import torch, torchvision
print("PyTorch:", torch.__version__, "torchvision:", torchvision.__version__)
assert torch.cuda.is_available(), "Enable a GPU before training."
print("GPU:", torch.cuda.get_device_name(0))
x = torch.randn(64, 64, device="cuda")
(x @ x).sum().backward() if x.requires_grad else (x @ x).sum().item()
torch.cuda.synchronize()
del x
print("CUDA computation passed.")


In [ ]:
import gdown, hashlib, json

DATASETS = {
    "severity": ("117lqrb69UR41ZeQXtTX895dE3sF8iTmo", "Dataset_Severity_V2.csv"),
    "poor_dilation": ("1hpCGfGdu-wjB2oi0xFNzb-pF4-mWjbWi", "Dataset_poordilation.csv"),
}
file_id, csv_name = DATASETS[TASK]
DATA_DIR = Path("/kaggle/working/data") / TASK
DATA_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = DATA_DIR / "images.zip"
EXTRACT_DIR = DATA_DIR / "extracted"
CSV_PATH = PROJECT_DIR / "metadata" / csv_name

if not ZIP_PATH.is_file():
    partial = ZIP_PATH.with_suffix(".part")
    result = gdown.download(id=file_id, output=str(partial), quiet=False)
    assert result and zipfile.is_zipfile(partial), "Download failed: check Drive sharing."
    partial.replace(ZIP_PATH)
assert zipfile.is_zipfile(ZIP_PATH)
with ZIP_PATH.open("rb") as f:
    zip_hash = hashlib.file_digest(f, "sha256").hexdigest()
marker = EXTRACT_DIR / "extraction.json"
if marker.exists():
    assert json.loads(marker.read_text())["sha256"] == zip_hash, "Dataset ZIP changed. Use a new extraction directory."
else:
    assert not EXTRACT_DIR.exists() or not any(EXTRACT_DIR.iterdir()), "Partial extraction exists; inspect or choose a new directory."
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        for name in z.namelist():
            assert (EXTRACT_DIR / name).resolve().is_relative_to(EXTRACT_DIR.resolve()), name
        z.extractall(EXTRACT_DIR)
    marker.write_text(json.dumps({"sha256": zip_hash}, indent=2))
print("Extracted:", EXTRACT_DIR)
print("Metadata snapshot:", CSV_PATH)


In [ ]:
import pandas as pd
from PIL import Image
sys.path.insert(0, str(PROJECT_DIR))
from metadata_utils import load_metadata, resolve_image
from dataset import PatientDataSplitter
from config import ModelConfig

metadata = load_metadata(TASK, CSV_PATH)
first = metadata.iloc[0]
# Locate folders bearing the exact original video ID, without renaming files.
candidates = {p.parent for p in EXTRACT_DIR.rglob(str(first.video_id)) if p.is_dir()}
valid_roots = []
for root in sorted(candidates):
    try:
        paths = [resolve_image(root, row.video_id, row.file_name)
                 for row in metadata.itertuples(index=False)]
        valid_roots.append((root, paths))
    except (ValueError, FileNotFoundError):
        pass
assert len(valid_roots) == 1, f"Expected one image root matching ALL rows; found {len(valid_roots)}. Inspect {EXTRACT_DIR}."
IMAGE_ROOT, image_paths = valid_roots[0]
assert len(set(image_paths)) == len(metadata), "Multiple records resolve to the same image."
for path in image_paths:
    with Image.open(path) as image:
        image.verify()

splits = PatientDataSplitter.split(metadata, ModelConfig.RANDOM_STATE)
for name, part in zip(("train", "val", "test"), splits):
    assert set(part.label) == set(metadata.label), f"Missing class in {name}"
    print(name, "frames:", len(part), "videos:", part.video_id.nunique(),
          "labels:", part.label.value_counts().sort_index().to_dict())
print("Image root:", IMAGE_ROOT)
print("All image and grouped-split checks passed.")


In [ ]:
OUTPUT_ROOT = Path("/kaggle/working/outputs") / RUN_NAME / TASK
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ENV = os.environ.copy()
prefix = "SEVERITY" if TASK == "severity" else "DILATION"
ENV[prefix + "_CSV_PATH"] = str(CSV_PATH)
ENV[prefix + "_IMG_DIR"] = str(IMAGE_ROOT)
ENV["OPTUNA_OUTPUT_DIR"] = str(OUTPUT_ROOT)
ENV["CUDA_VISIBLE_DEVICES"] = "0"
ENV["PYTHONUNBUFFERED"] = "1"

# Retain exact data/splits/environment used in this experiment.
metadata.to_csv(OUTPUT_ROOT / "metadata_used.csv", index=False)
for name, part in zip(("train", "val", "test"), splits):
    part.to_csv(OUTPUT_ROOT / f"{name}_split.csv", index=False)
(OUTPUT_ROOT / "environment.txt").write_text(subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"], text=True))
(OUTPUT_ROOT / "input_manifest.json").write_text(json.dumps({
    "task": TASK, "zip_sha256": zip_hash,
    "metadata_sha256": hashlib.sha256(CSV_PATH.read_bytes()).hexdigest(),
    "seed": ModelConfig.RANDOM_STATE, "image_root": str(IMAGE_ROOT),
}, indent=2))
print("Outputs:", OUTPUT_ROOT)


In [ ]:
selected = ModelConfig.MODELS if MODELS == ["all"] else MODELS
assert selected and len(selected) == len(set(selected))
assert set(selected) <= set(ModelConfig.MODELS), selected
for model in selected:
    result_dir = OUTPUT_ROOT / f"{model}_{TASK}"
    # This original project has no resume/skip-existing mechanism.
    assert not result_dir.exists(), f"Output already exists: {result_dir}. Select remaining models or use a new RUN_NAME."

for model in selected:
    print(f"Starting {model} / {TASK}", flush=True)
    subprocess.run([
        sys.executable, "-u", "main.py", "--task", TASK, "--model", model,
    ], cwd=PROJECT_DIR, env=ENV, check=True)
    # main.py catches training exceptions; verify its final report explicitly.
    report = OUTPUT_ROOT / f"{model}_{TASK}" / "README.md"
    assert report.is_file(), f"{model} did not finish. Inspect training output above."
print("All selected models completed.")


In [ ]:
print("Saved files:")
for p in sorted(OUTPUT_ROOT.rglob("*")):
    if p.is_file():
        print(p.relative_to(OUTPUT_ROOT), f"{p.stat().st_size / 1024**2:.2f} MB")
# Outputs remain under /kaggle/working/outputs for the saved notebook version.
